# 04 - Modelagem e avaliação

**Objetivo:** treinar, comparar e escolher o modelo de recomendação (Modelagem + Avaliação).

**Lê:** `data/clean/*.parquet`  
**Gera:** modelo final em `models/` para consulta da api 

In [35]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

proc = Path("../data/clean")
ratings = pd.read_parquet(proc/ "ratings_clean.parquet")
books = pd.read_parquet(proc/ "books_clean.parquet")
users = pd.read_parquet(proc/ "users_clean.parquet")



# 1- Fitro de esparsidade

Mantém usuários e livros com um minimo de avaliações

In [36]:
r_counts = ratings["User-ID"].value_counts()
for k in [1, 2, 3, 5, 10, 20]:
    u = r_counts[r_counts >= k]
    pct_users_count = len(u) / len(r_counts)
    pct_ratings = r_counts[r_counts >= k].sum() / r_counts.sum()
    print(f"min {k:>2}: mantém {pct_users_count:.1%} dos usuários e {pct_ratings:.1%} dos ratings")
    

        


    

min  1: mantém 100.0% dos usuários e 100.0% dos ratings
min  2: mantém 42.4% dos usuários e 89.8% dos ratings
min  3: mantém 29.7% dos usuários e 85.3% dos ratings
min  5: mantém 18.8% dos usuários e 78.7% dos ratings
min 10: mantém 9.7% dos usuários e 68.2% dos ratings
min 20: mantém 4.9% dos usuários e 56.7% dos ratings


Como a mediana de avaliações por usuário é 1, usar o segundo quartil como corte não removeria nada. Por isso foi adotado um mínimo fixo de 5 avaliações por usuário e por livro. Esse valor mantém 78.7% dos ratings mesmo removendo 82.2% dos usuários, e garante histórico mínimo para o modelo aprender preferências. A contrapartida é que o dataset filtrado representa usuários mais ativos que o usuário típico.

In [37]:
min_book, min_book = 5, 5

u = ratings['User-ID'].value_counts()
b = ratings['ISBN'].value_counts()
r = ratings[ratings['User-ID'].isin(u[u >= min_book].index) &
ratings['ISBN'].isin(b[b >= min_book].index)].copy()

print("Antes:", ratings.shape, "| Depois:", r.shape)
print("Usuários:", r['User-ID'].nunique(), "| Livros:", r['ISBN'].nunique())

Antes: (383843, 3) | Depois: (145890, 3)
Usuários: 12293 | Livros: 13769


## 2. Treino e teste

In [38]:
train, test = train_test_split(r, test_size=0.2, random_state=42)
print(train.shape, test.shape)

(116712, 3) (29178, 3)


# 3. Baselines


In [43]:
def avaliacao(y_true, y_pred, nome):
    r_msq = np.sqrt(mean_squared_error(y_true,y_pred))
    mae =mean_absolute_error(y_true,y_pred) 
    print(f"{nome} raiz do erro quadratico medio: {r_msq:.3f} erro medio absoluto {mae:.3f}")
    return {"modelo": nome, "raiz do erro quadratico medio:": r_msq, "erro medio absoluto:": mae}


resultados = []

#baseline 1: media global
media_global = train["Book-Rating"].mean()

resultados.append(avaliacao(test["Book-Rating"], np.full(len(test), media_global), "Media global:"))

#baseline 2: media por livro (fallback = media global)

media_livro = train.groupby("ISBN")["Book-Rating"].mean()
pred = test['ISBN'].map(media_livro).fillna(media_global)
resultados.append(avaliacao(test['Book-Rating'], pred, "Média por livro"))

Media global: raiz do erro quadratico medio: 1.788 erro medio absoluto 1.426
Média por livro raiz do erro quadratico medio: 1.828 erro medio absoluto 1.437
